# 🛡️ Threat IOC Analyzer
### Toolkit Forense y Ciberinteligencia para el Análisis Automatizado de Indicadores de Compromiso

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cristiangarcia913/threat-ioc-analyzer/blob/main/notebooks/Threat_IOC_Analyzer.ipynb)

**Autor:** Cristian García ([@Cristiangarcia913](https://github.com/Cristiangarcia913))  
**Versión:** 1.0.0 (Edición Profesional)  

---
Este cuaderno implementa el ciclo de vida forense completo para el tratamiento de **Indicadores de Compromiso (IOCs)** en investigaciones de incidentes de seguridad (DFIR) y Threat Intelligence:

1. **Desofuscación (Defang/Refang):** Reversión segura de técnicas de neutralización (hxxp://, [.], (dot), etc.).
2. **Normalización:** Estandarización de direcciones IP, dominios y URLs según estándares RFC.
3. **Clasificación:** Categorización precisa de IOCs (IPv4, IPv6, Dominio, URL, Hashes MD5/SHA).
4. **Resolución DNS:** Extracción automática de host y consulta de registros A, AAAA y resolución inversa PTR.
5. **Enriquecimiento RDAP:** Consulta estructurada a servicios de registro global (RIRs/ICANN) para extraer entidades, fechas de registro/expiración y contactos de abuso.

## 1. Configuración del Entorno
Instalamos y cargamos las dependencias necesarias. Este cuaderno es 100% compatible tanto con Google Colab como con entornos Jupyter locales.

In [ ]:
# Importaciones estándar y de red
import re
import os
import sys
import json
import socket
import ipaddress
from datetime import datetime
from urllib.parse import urlparse, unquote
import requests
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Detección dinámica de Google Colab
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import files
    print("✅ Ejecutando en Google Colab con soporte nativo de subida/descarga.")
else:
    print("✅ Ejecutando en entorno local Jupyter / Python.")

## 2. Motor Forense de IOCs (Clases y Funciones)

In [ ]:
# --- 1. DESOFUSCADOR ---
DEOBFUSCATION_RULES = [
    (r'hxxps?://', lambda m: 'https://' if 's' in m.group(0).lower() else 'http://'),
    (r'h\*\*ps?://', lambda m: 'https://' if 's' in m.group(0).lower() else 'http://'),
    (r'htxps?://', lambda m: 'https://' if 's' in m.group(0).lower() else 'http://'),
    (r'fxps?://', 'ftp://'),
    (r'\[://\]', '://'), (r'\[:\]', ':'), (r'//\]', '//'),
    (r'\[\.\]', '.'), (r'\(\.\)', '.'), (r'\{\.\}', '.'),
    (r'\[dot\]', '.'), (r'\(dot\)', '.'), (r'\{dot\}', '.'),
    (r'\\\.', '.'),
    (r'\[@\]', '@'), (r'\[at\]', '@'),
    (r'\[\s+\]', ''), (r'\[\]', '')
]

def desofuscar(ioc: str) -> str:
    if not isinstance(ioc, str): return ''
    res = ioc.strip()
    for pat, rep in DEOBFUSCATION_RULES:
        res = re.sub(pat, rep, res, flags=re.IGNORECASE)
    return res.strip()

def defang(ioc: str) -> str:
    if not isinstance(ioc, str): return ''
    t = re.sub(r'^https://', 'hxxps://', ioc.strip(), flags=re.IGNORECASE)
    t = re.sub(r'^http://', 'hxxp://', t, flags=re.IGNORECASE)
    return t.replace('.', '[.]').replace('@', '[@]')

# --- 2. NORMALIZADOR ---
class IOCNormalizer:
    def normalize(self, ioc: str) -> str:
        clean = ioc.strip()
        # Comprobar IP
        ip_m = re.match(r'^(\d{1,3}(?:\.\d{1,3}){3})(?::(\d{1,5}))?$', clean)
        if ip_m:
            try:
                ip_obj = ipaddress.IPv4Address(ip_m.group(1))
                port = f":{ip_m.group(2)}" if ip_m.group(2) else ""
                return f"{str(ip_obj)}{port}"
            except ValueError: pass
        # Comprobar URL
        if clean.startswith(('http://', 'https://', 'ftp://')) or '/' in clean:
            try:
                u = 'http://' + clean if not re.match(r'^[a-zA-Z]+://', clean) else clean
                p = urlparse(u)
                if p.netloc:
                    return f"{p.scheme.lower()}://{p.netloc.lower()}{unquote(p.path) or '/'}"
            except Exception: pass
        # Comprobar Dominio
        d = clean.lower().split(':')[0].rstrip('/')
        if re.match(r'^([a-z0-9]([a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,}$', d):
            return d
        return clean

# --- 3. CLASIFICADOR ---
class IOCClassifier:
    def classify(self, ioc: str) -> str:
        c = ioc.strip()
        if re.match(r'^[a-fA-F0-9]{32}$', c): return 'Hash (MD5)'
        if re.match(r'^[a-fA-F0-9]{40}$', c): return 'Hash (SHA-1)'
        if re.match(r'^[a-fA-F0-9]{64}$', c): return 'Hash (SHA-256)'
        if re.match(r'^(https?|ftp)://', c, re.IGNORECASE) or ('/' in c and not c.startswith('/')):
            return 'URL'
        if re.match(r'^\d{1,3}(\.\d{1,3}){3}:\d{1,5}$', c): return 'IPv4:Port'
        try:
            addr = ipaddress.ip_address(c)
            return 'IPv4' if isinstance(addr, ipaddress.IPv4Address) else 'IPv6'
        except ValueError: pass
        if re.match(r'^([a-zA-Z0-9]([a-zA-Z0-9-]{0,61}[a-zA-Z0-9])?\.)+[a-zA-Z]{2,}$', c):
            return 'Dominio'
        return 'Desconocido'

# --- 4. RESOLUTOR DNS ---
def resolver_dns(ioc: str, timeout: float = 5.0) -> dict:
    res = {'target': None, 'A': [], 'AAAA': [], 'PTR': None, 'status': 'ok'}
    target = ioc.strip()
    if target.startswith(('http://', 'https://')):
        target = urlparse(target).hostname or target
    if '/' in target: target = target.split('/')[0]
    if ':' in target and not target.startswith('['): target = target.split(':')[0]
    res['target'] = target
    
    old_timeout = socket.getdefaulttimeout()
    socket.setdefaulttimeout(timeout)
    try:
        try:
            ip = ipaddress.ip_address(target)
            try:
                ptr_val = socket.gethostbyaddr(str(ip))[0]
                res['PTR'] = ptr_val
            except Exception:
                res['PTR'] = 'Sin registro PTR'
        except ValueError:
            try:
                res['A'] = socket.gethostbyname_ex(target)[2]
            except Exception: pass
            try:
                ai = socket.getaddrinfo(target, None, socket.AF_INET6)
                res['AAAA'] = list(set([r[4][0] for r in ai]))
            except Exception: pass
    finally:
        socket.setdefaulttimeout(old_timeout)
    return res

# --- 5. CLIENTE RDAP ---
def consultar_rdap(ioc: str, timeout: float = 10.0) -> dict:
    target = ioc.strip()
    if target.startswith(('http://', 'https://')):
        target = urlparse(target).hostname or target
    if '/' in target: target = target.split('/')[0]
    if ':' in target and not target.startswith('['): target = target.split(':')[0]
    
    try:
        ipaddress.ip_address(target)
        q_type = 'ip'
    except ValueError:
        q_type = 'domain'
        
    url = f'https://rdap.org/{q_type}/{target}'
    try:
        r = requests.get(url, headers={'Accept': 'application/rdap+json'}, timeout=timeout)
        if r.status_code == 200:
            d = r.json()
            name = d.get('name') or d.get('ldhName') or 'N/A'
            handle = d.get('handle') or 'N/A'
            exp = 'N/A'
            for ev in d.get('events', []):
                if ev.get('eventAction') == 'expiration':
                    exp = ev.get('eventDate', 'N/A')
            return {'handle': handle, 'name': name, 'expiration': exp, 'status': 'success'}
        return {'status': f'HTTP {r.status_code}'}
    except Exception as e:
        return {'status': f'Error: {str(e)}'}

normalizer = IOCNormalizer()
classifier = IOCClassifier()
print('🚀 Motor Forense cargado correctamente.')

## 3. Interfaz Interactiva de Análisis (UI con Widgets)
Permite analizar IOCs en tiempo real con una interfaz moderna y amigable.

In [ ]:
# Componentes UI
txt_input = widgets.Text(
    placeholder='Introduce un IOC (ej: hxxp://evil[.]com/malware.exe o 185[.]203[.]118[.]55)',
    description='IOC:',
    layout=widgets.Layout(width='80%')
)
btn_run = widgets.Button(
    description='Analizar IOC 🚀',
    button_style='success',
    icon='search'
)
output_area = widgets.Output()

def on_click_analyze(b):
    output_area.clear_output(wait=True)
    with output_area:
        raw = txt_input.value.strip()
        if not raw:
            print('⚠️ Por favor ingresa un IOC.')
            return
        
        print(f'🔍 Analizando indicador: {raw}\n')
        # 1. Desofuscar
        deob = desofuscar(raw)
        print(f'1. Desofuscado   : {deob}')
        # 2. Normalizar
        norm = normalizer.normalize(deob)
        print(f'2. Normalizado   : {norm}')
        # 3. Clasificar
        tipo = classifier.classify(norm)
        print(f'3. Clasificación : {tipo}\n')
        
        # 4. DNS
        print('⏳ Consultando DNS...')
        dns = resolver_dns(norm)
        if dns.get('A'): print(f'   • IPv4 (A)    : {", ".join(dns["A"])}')
        if dns.get('AAAA'): print(f'   • IPv6 (AAAA) : {", ".join(dns["AAAA"])}')
        if dns.get('PTR'): print(f'   • PTR         : {dns["PTR"]}')
        
        # 5. RDAP
        print('\n⏳ Consultando RDAP...')
        rdap = consultar_rdap(norm)
        if rdap.get('status') == 'success':
            print(f'   • Entidad     : {rdap.get("name")}')
            print(f'   • Handle      : {rdap.get("handle")}')
            print(f'   • Expiración  : {rdap.get("expiration")}')
        else:
            print(f'   • Estado RDAP : {rdap.get("status")}')
        print('\n✅ Análisis forense completado con éxito.')

btn_run.on_click(on_click_analyze)
display(widgets.VBox([txt_input, btn_run, output_area]))

## 4. Procesamiento en Lote (Batch File Processing)
Procesa una lista completa de IOCs desde un archivo de texto (como la muestra de Mirai) y genera un reporte descargable.

In [ ]:
# Muestra de prueba integrada basada en la Botnet Mirai
MIRAI_SAMPLES = """
hxxp://93[.]123[.]85[.]239/mirai[.]m68k
hxxp://93[.]123[.]85[.]239/mirai[.]arm
hxxp://181[.]214[.]39[.]189/mirai[.]arm4
hxxp://91[.]208[.]206[.]217/hidden[.]sh
hxxp://5[.]175[.]237[.]74/GuruITDDoS/RpcSecurity[.]x86
miraicnc[.]ddns[.]net
rebirthbot[.]icu
dreamproxy[.]cc
gg[.]elrooted[.]com
45[.]77[.]89[.]12
185[.]203[.]118[.]55
176[.]65[.]149[.]34
196[.]251[.]80[.]5:3778
""".strip().splitlines()

print(f"📊 Procesando {len(MIRAI_SAMPLES)} IOCs de la muestra Mirai...\n")
resultados = []

for ioc in MIRAI_SAMPLES:
    d = desofuscar(ioc)
    n = normalizer.normalize(d)
    t = classifier.classify(n)
    resultados.append({'original': ioc, 'desofuscado': d, 'normalizado': n, 'tipo': t})
    print(f"• {ioc:<50} -> {t} ({n})")

print(f"\n✅ {len(resultados)} IOCs procesados y clasificados exitosamente.")